# Skeleton-based action recognition using TAO PoseClassificationNet

Transfer learning is the process of transferring learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. 

Train Adapt Optimize (TAO) Toolkit  is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

<img align="center" src="https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png" width="1080">


## Learning Objectives

In this notebook, you will learn how to leverage the simplicity and convenience of TAO to:

* Train a model for skeleton-based action recognition on the [Kinetics](https://deepmind.com/research/open-source/kinetics) dataset.
* Evaluate the trained model.
* Run Inference on the trained model.
* Export the trained model to an .onnx file (encrypted ONNX model) for deployment to DeepStream or TensorRT.
* Convert the pose data from [deepstream-bodypose-3d](https://github.com/NVIDIA-AI-IOT/deepstream_reference_apps/tree/master/deepstream-bodypose-3d) to skeleton arrays for inference.

At the end of this notebook, you will have generated a trained and optimized `PoseClassification` model, 
which you may deploy with this [end-to-end sample](https://github.com/NVIDIA-AI-IOT/tao-toolkit-triton-apps) with Triton.

## Table of Contents

This notebook shows an example usecase of PoseClassificationNet using Train Adapt Optimize (TAO) Toolkit.

0. [Set up env variables and map drives](#head-0)
1. [Installing the TAO launcher](#head-1)
2. [Prepare dataset and pre-trained model](#head-2)
3. [Provide training specification](#head-3)
4. [Run TAO training](#head-4)
5. [Evaluate trained models](#head-5)
6. [Inferences](#head-6)
7. [Deploy](#head-7)
8. [Convert pose data](#head-8)


## 0. Set up env variables and map drives <a class="anchor" id="head-0"></a>

When using the purpose-built pretrained models from NGC, please make sure to set the `$KEY` environment variable to the key as mentioned in the model overview. Failing to do so, can lead to errors when trying to load them as pretrained models.

The TAO launcher uses docker containers under the hood, and **for our data and results directory to be visible to the docker, they need to be mapped**. The launcher can be configured using the config file `~/.tao_mounts.json`. Apart from the mounts, you can also configure additional options like the Environment Variables and amount of Shared Memory available to the TAO launcher. <br>

`IMPORTANT NOTE:` The code below creates a sample `~/.tao_mounts.json`  file. Here, we can map directories in which we save the data, specs, results and cache. You should configure it for your specific case so these directories are correctly visible to the docker container.


In [1]:
import os

# Please define this local project directory that needs to be mapped to the TAO docker session.
%env LOCAL_PROJECT_DIR=/home/isaacsim/Documents/Lucija/tao-experiments

os.environ["HOST_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data", "poseclassificationnet")
os.environ["HOST_RESULTS_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "poseclassificationnet")

# Set this path if you don't run the notebook from the samples directory.
# %env NOTEBOOK_ROOT=/path/to/local/tao-experiments/pose_classification_net
# The sample spec files are present in the same path as the downloaded samples.
os.environ["HOST_SPECS_DIR"] = os.path.join(
    os.getenv("NOTEBOOK_ROOT", os.getcwd()),
    "specs"
)
os.environ["PROJECT_DIR"]="/home/isaacsim/Documents/Lucija/tao-tutorials/notebooks/tao_launcher_starter_kit"

# Set your encryption key, and use the same key for all commands
%env KEY = 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

env: LOCAL_PROJECT_DIR=/home/isaacsim/Documents/Lucija/tao-experiments
env: KEY=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


In [2]:
! mkdir -p $HOST_DATA_DIR
! mkdir -p $HOST_SPECS_DIR
! mkdir -p $HOST_RESULTS_DIR

In [3]:
!cat ~/.tao_mounts.json

{
    "Mounts": [
        {
            "source": "/home/isaacsim/Documents/Lucija/tao-experiments",
            "destination": "/workspace/tao-experiments"
        },
        {
            "source": "/home/isaacsim/Documents/Lucija/tao-experiments/data/poseclassificationnet",
            "destination": "/data"
        },
        {
            "source": "/home/isaacsim/Documents/Lucija/tao-tutorials/notebooks/tao_launcher_starter_kit/pose_classification_net/specs",
            "destination": "/specs"
        },
        {
            "source": "/home/isaacsim/Documents/Lucija/tao-experiments/poseclassificationnet",
            "destination": "/results"
        }
    ],
    "DockerOptions": {
        "user": "1000:1000",
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
        }
    }
}

In [4]:
# Mapping up the local directories to the TAO docker.
import json
import os
mounts_file = os.path.expanduser("~/.tao_mounts.json")
tlt_configs = {
   "Mounts":[
       # Mapping the data directory
       {
           "source": os.environ["LOCAL_PROJECT_DIR"],
           "destination": "/workspace/tao-experiments"
       },
       {
           "source": os.environ["HOST_DATA_DIR"],
           "destination": "/data"
       },
       {
           "source": os.environ["HOST_SPECS_DIR"],
           "destination": "/specs"
       },
       {
           "source": os.environ["HOST_RESULTS_DIR"],
           "destination": "/results"
       }
   ],
   "DockerOptions": {
       "user": "1000:1000",
        "shm_size": "16G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
         }
   }
}
# Writing the mounts file.
with open(mounts_file, "w") as mfile:
    json.dump(tlt_configs, mfile, indent=4)

## 1. Installing the TAO launcher <a class="anchor" id="head-1"></a>
The TAO launcher is a python package distributed as a python wheel listed in PyPI. You may install the launcher by executing the following cell.

Please note that TAO Toolkit recommends users to run the TAO launcher in a virtual env with python 3.6.9. You may follow the instruction in this [page](https://virtualenvwrapper.readthedocs.io/en/latest/install.html) to set up a python virtual env using the `virtualenv` and `virtualenvwrapper` packages. Once you have setup virtualenvwrapper, please set the version of python to be used in the virtual env by using the `VIRTUALENVWRAPPER_PYTHON` variable. You may do so by running

```sh
export VIRTUALENVWRAPPER_PYTHON=/path/to/bin/python3.x
```
where x >= 6 and <= 8

We recommend performing this step first and then launching the notebook from the virtual environment. In addition to installing TAO python package, please make sure of the following software requirements:
* python >=3.7, <=3.10.x
* docker-ce > 19.03.5
* docker-API 1.40
* nvidia-container-toolkit > 1.3.0-1
* nvidia-container-runtime > 3.4.0-1
* nvidia-docker2 > 2.5.0-1
* nvidia-driver > 455+

Once you have installed the pre-requisites, please log in to the docker registry nvcr.io by following the command below

```sh
docker login nvcr.io
```

You will be triggered to enter a username and password. The username is `$oauthtoken` and the password is the API key generated from `ngc.nvidia.com`. Please follow the instructions in the [NGC setup guide](https://docs.nvidia.com/ngc/ngc-overview/index.html#generating-api-key) to generate your own API key.

Please note that TAO Toolkit recommends users to run the TAO launcher in a virtual env with python >=3.6.9. You may follow the instruction in this [page](https://virtualenvwrapper.readthedocs.io/en/latest/install.html) to set up a python virtual env using the virtualenv and virtualenvwrapper packages.

In [5]:
# SKIP this step IF you have already installed the TAO launcher.
!pip3 install nvidia-tao

In [5]:
# View the versions of the TAO launcher
!tao info

Configuration of the TAO Toolkit Instance
task_group: ['model', 'dataset', 'deploy']
format_version: 3
toolkit_version: 6.26.3
published_date: 03/20/2026


## 2. Prepare dataset and pre-trained model <a class="anchor" id="head-2"></a>
 We will be using the [Kinetics](https://deepmind.com/research/open-source/kinetics) dataset for the tutorial. Download the pre-processed data of Kinetics-Skeleton [here](https://drive.google.com/uc?id=1dmzCRQsFXJ18BlXj1G9sbDnsclXIdDdR) and extract them first: 

In [3]:
# download the dataset.
!pip3 install -U gdown
!gdown https://drive.google.com/uc?id=1dmzCRQsFXJ18BlXj1G9sbDnsclXIdDdR -O $HOST_DATA_DIR/st-gcn-processed-data.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gdown]
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1dmzCRQsFXJ18BlXj1G9sbDnsclXIdDdR

but Gdown can't. Please check connections and permissions.


In [ ]:
# extract the files
!unzip -o $HOST_DATA_DIR/st-gcn-processed-data.zip -d $HOST_DATA_DIR
!mv $HOST_DATA_DIR/data/Kinetics/kinetics-skeleton $HOST_DATA_DIR/kinetics
!rm -r $HOST_DATA_DIR/data
!rm $HOST_DATA_DIR/st-gcn-processed-data.zip

In [ ]:
# verify
!ls -l $HOST_DATA_DIR/kinetics

In [ ]:
# Install required dependencies from the notebook.
!pip3 install Cython==0.29.36
!pip3 install -r $PROJECT_DIR/deps/requirements-pip.txt

In [ ]:
# select actions
import os
import pickle
import numpy as np

data_dir = os.path.join(os.environ["HOST_DATA_DIR"], "kinetics")

# front_raises: 134
# pull_ups: 255
# clean_and_jerk: 59
# presenting_weather_forecast: 254
# deadlifting: 88
selected_actions = {
    134: 0,
    255: 1,
    59: 2,
    254: 3,
    88: 4
}

def select_actions(selected_actions, data_dir, split_name):
    """Select a subset of actions and their corresponding labels.
    
    Args:
        selected_actions (dict): Map from selected class IDs to new class IDs.
        data_dir (str): Path to the directory of data arrays (.npy) and labels (.pkl).
        split_name (str): Name of the split to be processed, e.g., "train" and "val".
        
    Returns:
        No explicit returns
    """
    data_path = os.path.join(data_dir, f"{split_name}_data.npy")
    label_path = os.path.join(data_dir, f"{split_name}_label.pkl")

    data_array = np.load(file=data_path)
    with open(label_path, "rb") as label_file:
        labels = pickle.load(label_file)

    assert(len(labels) == 2)
    assert(data_array.shape[0] == len(labels[0]))
    assert(len(labels[0]) == len(labels[1]))

    print(f"No. total samples for {split_name}: {data_array.shape[0]}")

    selected_indices = []
    for i in range(data_array.shape[0]):
        if labels[1][i] in selected_actions.keys():
            selected_indices.append(i)

    data_array = data_array[selected_indices, :, :, :, :]
    selected_sample_names = [labels[0][x] for x in selected_indices]
    selected_labels = [selected_actions[labels[1][x]] for x in selected_indices]
    labels = (selected_sample_names, selected_labels)

    print(f"No. selected samples for {split_name}: {data_array.shape[0]}")

    np.save(file=data_path, arr=data_array, allow_pickle=False)
    with open(label_path, "wb") as label_file:
        pickle.dump(labels, label_file, protocol=4)

select_actions(selected_actions, data_dir, "train")
select_actions(selected_actions, data_dir, "val")

We also provide scripts to process the NVIDIA dataset generated by [deepstream-bodypose-3d](https://github.com/NVIDIA-AI-IOT/deepstream_reference_apps/tree/master/deepstream-bodypose-3d). The following cells for processing the NVIDIA dataset is `Optional`.

`OPTIONAL:` Download the NVIDIA dataset and extract the files.

In [ ]:
# # Download the dataset
# !pip3 install -U gdown
# !gdown https://drive.google.com/uc?id=1GhSt53-7MlFfauEZ2YkuzOaZVNIGo_c- -O $HOST_DATA_DIR/data_3dbp_nvidia.zip

In [ ]:
# # Extract the files
# !mkdir -p $HOST_DATA_DIR/nvidia
# !unzip $HOST_DATA_DIR/data_3dbp_nvidia.zip -d $HOST_DATA_DIR/nvidia
# !rm $HOST_DATA_DIR/data_3dbp_nvidia.zip

In [ ]:
# # Verify
# !ls -l $HOST_DATA_DIR/nvidia

`OPTIONAL:` Download the pretrained model from NGC. We will use NGC CLI to get the data and model. For more details, go to https://ngc.nvidia.com and click the SETUP on the navigation bar.

In [ ]:
# # Installing NGC CLI on the local machine.
# ## Download and install
# import os
# import platform

# # Detect system architecture and set appropriate NGC CLI package
# arch = platform.machine().lower()
# if arch == "x86_64" or arch == "amd64":
#     cli_package = "ngccli_linux.zip"
# elif arch == "aarch64" or arch == "arm64":
#     cli_package = "ngccli_arm64.zip"
# else:
#     raise ValueError(f"Unsupported architecture: {arch}. Supported architectures are x86_64/amd64 and aarch64/arm64.")

# print(f"Detected architecture: {arch}")
# print(f"Downloading NGC CLI package: {cli_package}")

# os.environ["CLI"] = cli_package

# # Remove any previously existing CLI installations
# !rm -rf $HOST_RESULTS_DIR/ngccli/*
# !wget "https://ngc.nvidia.com/downloads/$CLI" -P $HOST_RESULTS_DIR/ngccli
# !unzip -u "$HOST_RESULTS_DIR/ngccli/$CLI" -d $HOST_RESULTS_DIR/ngccli/
# !rm $HOST_RESULTS_DIR/ngccli/*.zip 
# os.environ["PATH"]="{}/ngccli/ngc-cli:{}".format(os.getenv("HOST_RESULTS_DIR", ""), os.getenv("PATH", ""))

In [ ]:
# !ngc registry model list nvidia/tao/poseclassificationnet:*

In [ ]:
# !mkdir -p $HOST_RESULTS_DIR/pretrained

In [ ]:
# # Pull pretrained model from NGC 
# !ngc registry model download-version "nvidia/tao/poseclassificationnet:trainable_v1.0" --dest $HOST_RESULTS_DIR/pretrained

In [ ]:
# print("Check that model is downloaded into dir.")
# !ls -l $HOST_RESULTS_DIR/pretrained/poseclassificationnet_vtrainable_v1.0

## 3. Provide training specification <a class="anchor" id="head-3"></a>

We provide specification files to configure the training parameters including:

* model: configure the model setting
    * model_type: type of model, ST-GCN
    * pretrained_model_path: path for the input model
    * input_channels: number of input channels
    * dropout: probability to drop the hidden units
    * graph_layout: type of graph layout, nvidia/openpose/human3.6m/ntu-rgb+d/ntu_edge/coco
    * graph_strategy: type of graph strategy, uniform/distance/spatial
    * edge_importance_weighting: enabling edge importance weighting
* dataset: configure the dataset and augmentation methods
    * train_dataset: paths for the training data and label file
    * val_dataset: paths for the validation data and label file
    * num_classes: number of classes
    * label_map: map from labels to class IDs
    * random_choose: enabling randomly choosing a portion of the input sequence
    * random_move: enabling randomly moving the input sequence
    * window_size: length of the output sequence
    * batch_size: number of arrays in 1 batch
    * num_workers: number of workers to do data loading
* train: configure the training hyperparameters
    * optim: configure optimizer
    * num_epochs: number of epochs
    * checkpoint_interval: enabling how often to store models
    * grad_clip: enabling gradient clipping

Please refer to the TAO documentation about PoseClassificationNet to get all the parameters that are configurable.

In [6]:
!cat $HOST_SPECS_DIR/experiment_lucija.yaml

results_dir: "/results/nvidia"
encryption_key: nvidia_tao
model:
  model_type: ST-GCN
  pretrained_model_path: ""
  input_channels: 3
  dropout: 0.5
  graph_layout: "nvidia"
  graph_strategy: "spatial"
  edge_importance_weighting: True
dataset:
  train_dataset:
    data_path: "/data/nvidia/dataset_w3/train_data.npy"
    label_path: "/data/nvidia/dataset_w3/train_label.pkl"
  val_dataset:
    data_path: "/data/nvidia/dataset_without2/val_data.npy"
    label_path: "/data/nvidia/dataset_without2/val_label.pkl"
  num_classes: 4
  label_map:
    sitting: 0
    standing: 1
    walking: 2
    raising_hand: 3
  batch_size: 16
  num_workers: 1
train:
  optim:
    lr: 0.1
    momentum: 0.9
    nesterov: True
    weight_decay: 0.0001
    lr_scheduler: "MultiStep"
    lr_steps:
    - 10
    - 60
    lr_decay: 0.1
  num_epochs: 70
  num_gpus: 1
  checkpoint_interval: 5
dataset_convert:
  pose_type: "3dbp"
  num_joints: 34
  input_width: 1920
  input_height: 1080
  focal_length: 1385.9
  sequence_le

## 4. Run TAO training <a class="anchor" id="head-4"></a>
* Provide the sample spec file and the output directory location for models.
* WARNING: Training will take several hours or one day to complete.

In [5]:
# NOTE: The following paths are set from the perspective of the TAO Docker.

# The data is saved here
%env DATA_DIR = /data
%env SPECS_DIR = /specs
%env RESULTS_DIR = /results

env: DATA_DIR=/data
env: SPECS_DIR=/specs
env: RESULTS_DIR=/results


### 4.1 Train Kinetics model

We will train a Kinetics model from scratch.

In [8]:
print("Train model")
!tao model pose_classification train \
                  -e $SPECS_DIR/experiment_lucija.yaml \
                  results_dir=$RESULTS_DIR/lucija \
                  encryption_key=$KEY

Train model
2026-06-02 02:23:28,930 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-06-02 02:23:28,988 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:6.26.3-pyt
2026-06-02 02:23:29,586 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 316: Printing tty value True
[2026-06-02 06:23:32,017 - TAO Toolkit - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO] Logging configured at level: INFO
INFO: Logging configured at level: INFO
sys:1: UserWarning: 
'experiment_lucija.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
/usr/local/lib/python3.12/dist-packages/nvidia_tao_pytorch/core/hydra/hydra_runner.py:110: UserWarning: 
'experiment_lucija.yaml' is valid

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1539: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /results/lucija/train exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
  | Name           | Type

Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 84.47it/s] 
                                                                      
Epoch 3: 100%|██████████| 8/8 [00:00<00:00, 14.15it/s, v_num=1, train_loss_step=0.785, val_loss=0.696, train_loss_epoch=0.718]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 84.11it/s] 
                                                                      
Epoch 4: 100%|██████████| 8/8 [00:00<00:00, 14.03it/s, v_num=1, train_loss_step=0.877, val_loss=0.842, train_loss_epoch=0.869]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 86.48it/s] 
                                                                      
Epoch 5: 100%|██████████| 8/8 [00:00<00:00, 14.09it/s, v_num=1, train_loss_step=0.735, val_loss=0.669, train_loss_epoch=0.819]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00

Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 86.18it/s] 
                                                                      
Epoch 28: 100%|██████████| 8/8 [00:00<00:00, 14.39it/s, v_num=1, train_loss_step=0.580, val_loss=0.591, train_loss_epoch=0.398]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 87.83it/s] 
                                                                      
Epoch 29: 100%|██████████| 8/8 [00:00<00:00, 14.32it/s, v_num=1, train_loss_step=0.366, val_loss=0.558, train_loss_epoch=0.456]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 86.38it/s] 
                                                                      
Epoch 30: 100%|██████████| 8/8 [00:00<00:00, 14.35it/s, v_num=1, train_loss_step=0.387, val_loss=0.561, train_loss_epoch=0.388]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00

Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 88.23it/s] 
                                                                      
Epoch 40: 100%|██████████| 8/8 [00:00<00:00, 14.24it/s, v_num=1, train_loss_step=0.342, val_loss=0.541, train_loss_epoch=0.378]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 83.46it/s] 
                                                                      
Epoch 41: 100%|██████████| 8/8 [00:00<00:00, 14.03it/s, v_num=1, train_loss_step=0.338, val_loss=0.529, train_loss_epoch=0.341]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 83.48it/s] 
                                                                      
Epoch 42: 100%|██████████| 8/8 [00:00<00:00, 14.13it/s, v_num=1, train_loss_step=0.310, val_loss=0.520, train_loss_epoch=0.399]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00

Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 84.00it/s] 
                                                                      
Epoch 65: 100%|██████████| 8/8 [00:00<00:00, 14.17it/s, v_num=1, train_loss_step=0.377, val_loss=0.541, train_loss_epoch=0.273]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 84.05it/s] 
                                                                      
Epoch 66: 100%|██████████| 8/8 [00:00<00:00, 14.06it/s, v_num=1, train_loss_step=0.399, val_loss=0.552, train_loss_epoch=0.328]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 87.73it/s] 
                                                                      
Epoch 67: 100%|██████████| 8/8 [00:00<00:00, 14.12it/s, v_num=1, train_loss_step=0.232, val_loss=0.531, train_loss_epoch=0.294]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 4/4 [00

In [ ]:
# print("Train model using multiple (2) GPUs")
# !tao model pose_classification train \
#                   -e $SPECS_DIR/experiment_kinetics.yaml \
#                   results_dir=$RESULTS_DIR/kinetics \
#                   encryption_key=$KEY \
#                   train.gpu_ids=[0,1]

In [9]:
print('Encrypted checkpoints:')
print('---------------------')
!ls -ltrh $HOST_RESULTS_DIR/lucija/train

Encrypted checkpoints:
---------------------
total 417M
drwxr-xr-x 3 isaacsim isaacsim 4.0K Jun  2 02:23 lightning_logs
-rw-r--r-- 1 isaacsim isaacsim 2.3K Jun  2 02:23 experiment.yaml
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:23 model_epoch_004_step_00040.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:23 model_epoch_009_step_00080.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:23 model_epoch_014_step_00120.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_019_step_00160.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_024_step_00200.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_029_step_00240.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_034_step_00280.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_039_step_00320.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_044_step_00360.pth
-rw-r--r-- 1 isaacsim isaacsim  30M Jun  2 02:24 model_epoch_049_step_00400.pth
-rw-r--r-- 1 is

In [10]:
# You can set NUM_EPOCH to the epoch corresponding to any saved checkpoint
# %env NUM_EPOCH=029

# Get the name of the checkpoint corresponding to your set epoch
# tmp=!ls $HOST_RESULTS_DIR/kinetics/train/*.pth | grep epoch_$NUM_EPOCH
# %env CHECKPOINT={tmp[0]}

# Or get the latest checkpoint
os.environ["CHECKPOINT"] = os.path.join(os.getenv("HOST_RESULTS_DIR"), "lucija/train/pc_model_latest.pth")

print('Rename a trained model: ')
print('---------------------')
!cp $CHECKPOINT $HOST_RESULTS_DIR/lucija/train/lucija_model.tlt
!ls -ltrh $HOST_RESULTS_DIR/lucija/train/lucija_model.tlt

Rename a trained model: 
---------------------
-rw-r--r-- 1 isaacsim isaacsim 30M Jun  2 02:25 /home/isaacsim/Documents/Lucija/tao-experiments/poseclassificationnet/lucija/train/lucija_model.tlt


### `OPTIONAL` 4.2 Train NVIDIA model

In [ ]:
# print("Train model from scratch")
# !tao model pose_classification train \
#                   -e $SPECS_DIR/experiment_nvidia.yaml \
#                   results_dir=$RESULTS_DIR/nvidia \
#                   encryption_key=$KEY

In [ ]:
# print("Train model from scratch using multiple (2) GPUs")
# !tao model pose_classification train \
#                   -e $SPECS_DIR/experiment_nvidia.yaml \
#                   results_dir=$RESULTS_DIR/nvidia \
#                   encryption_key=$KEY \
#                   train.gpu_ids=[0,1]

We provide pre-trained ST-GCN model trained on the NVIDIA dataset. With the pre-trained model, we can even get better accuracy with less epochs.

In [ ]:
# print("To resume training from a checkpoint, set the model.pretrained_model_path option to be the .tlt you want to resume from")
# print("remember to remove the `=` in the checkpoint's file name")
# !tao model pose_classification train \
#                   -e $SPECS_DIR/experiment_nvidia.yaml \
#                   results_dir=$RESULTS_DIR/nvidia \
#                   encryption_key=$KEY \
#                   model.pretrained_model_path=

In [ ]:
# print('Encrypted checkpoints:')
# print('---------------------')
# !ls -ltrh $HOST_RESULTS_DIR/nvidia/train

In [ ]:
# You can set NUM_EPOCH to the epoch corresponding to any saved checkpoint
# %env NUM_EPOCH=029

# Get the name of the checkpoint corresponding to your set epoch
# tmp=!ls $HOST_RESULTS_DIR/nvidia/train/*.pth | grep epoch_$NUM_EPOCH
# %env CHECKPOINT={tmp[0]}

# Or get the latest checkpoint
# os.environ["CHECKPOINT"] = os.path.join(os.getenv("HOST_RESULTS_DIR"), "nvidia/train/pc_model_latest.pth")

# print('Rename a trained model: ')
# print('---------------------')
# !cp $CHECKPOINT $HOST_RESULTS_DIR/nvidia/train/nvidia_model.tlt
# !ls -ltrh $HOST_RESULTS_DIR/nvidia/train/nvidia_model.tlt

## 5. Evaluate trained models <a class="anchor" id="head-5"></a>
Evaluate trained model.

In [12]:
!tao model pose_classification evaluate \
                    -e $SPECS_DIR/experiment_lucija.yaml \
                    results_dir=$RESULTS_DIR/lucija \
                    encryption_key=$KEY \
                    evaluate.checkpoint=$RESULTS_DIR/lucija/train/lucija_model.tlt \
                    evaluate.test_dataset.data_path=$DATA_DIR/nvidia/dataset_w3/val_data.npy \
                    evaluate.test_dataset.label_path=$DATA_DIR/nvidia/dataset_w3/val_label.pkl

2026-06-02 02:25:49,401 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-06-02 02:25:49,457 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:6.26.3-pyt
2026-06-02 02:25:50,042 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 316: Printing tty value True
[2026-06-02 06:25:52,495 - TAO Toolkit - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO] Logging configured at level: INFO
2026-06-02 06:26:00,512 - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO - Logging configured at level: INFO
sys:1: UserWarning: 
'experiment_lucija.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
/usr/local/lib/python3.12/dist-packages/nvidia_tao_pytorch/core/hy

2026-06-02 06:26:04,650 - [TAO Toolkit] - INFO - Execution status: PASS (entrypoint.py:376)
2026-06-02 06:26:04,650 - [TAO Toolkit] - INFO - Execution status: PASS (entrypoint.py:376)
[2026-06-02 06:26:04,650 - TAO Toolkit - TAO Toolkit - INFO] Execution status: PASS
2026-06-02 02:26:05,683 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 381: Stopping container.


## 6. Inferences <a class="anchor" id="head-6"></a>
In this section, we run the pose classification inference tool to generate inferences with the trained models and save the results under `$RESULTS_DIR`. 

In [9]:
!tao model pose_classification inference \
                    -e $SPECS_DIR/experiment_lucija.yaml \
                    results_dir=$RESULTS_DIR/lucija \
                    encryption_key=$KEY \
                    inference.checkpoint=$RESULTS_DIR/lucija/train/lucija_model.tlt \
                    inference.output_file=$RESULTS_DIR/lucija/inference/inference_2.txt \
                    inference.test_dataset.data_path=$DATA_DIR/nvidia/proba_zavrsni/output2.npy

2026-06-08 09:22:37,171 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-06-08 09:22:37,227 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:6.26.3-pyt
2026-06-08 09:22:38,147 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 316: Printing tty value True
[2026-06-08 13:22:40,564 - TAO Toolkit - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO] Logging configured at level: INFO
2026-06-08 13:22:48,365 - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO - Logging configured at level: INFO
sys:1: UserWarning: 
'experiment_lucija.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
/usr/local/lib/python3.12/dist-packages/nvidia_tao_pytorch/core/hy

## 7. Deploy <a class="anchor" id="head-7"></a>
Export the model to encrypted ONNX model.

In [7]:
!tao model pose_classification export \
                   -e $SPECS_DIR/experiment_lucija.yaml \
                   results_dir=$RESULTS_DIR/lucija \
                   encryption_key=$KEY \
                   export.checkpoint=$RESULTS_DIR/lucija/train/lucija_model.tlt \
                   export.onnx_file=$RESULTS_DIR/lucija/export/lucija_model.onnx

2026-06-08 02:51:56,324 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-06-08 02:51:56,386 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:6.26.3-pyt
2026-06-08 02:51:57,331 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 316: Printing tty value True
[2026-06-08 06:51:59,759 - TAO Toolkit - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO] Logging configured at level: INFO
2026-06-08 06:52:05,312 - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO - Logging configured at level: INFO
sys:1: UserWarning: 
'experiment_lucija.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
/usr/local/lib/python3.12/dist-packages/nvidia_tao_pytorch/core/hy

In [8]:
print('Exported model:')
print('------------')
!ls -lth $HOST_RESULTS_DIR/lucija/export

Exported model:
------------
total 13M
-rw-r--r-- 1 isaacsim isaacsim  390 Jun  8 02:52 status.json
-rw-r--r-- 1 isaacsim isaacsim 2.3K Jun  8 02:52 experiment.yaml
-rw-r--r-- 1 isaacsim isaacsim  13M Jun  2 02:27 lucija_model.onnx


You may continue by deploying the exported model to [Triton Inference Server](https://developer.nvidia.com/nvidia-triton-inference-server). Please refer to the [TAO Toolkit Triton Apps](https://github.com/NVIDIA-AI-IOT/tao-toolkit-triton-apps), where a sample for end-to-end inference from video is also provided. 

## `OPTIONAL` 8. Convert pose data <a class="anchor" id="head-8"></a>
Convert the JSON pose data from [deepstream-bodypose-3d](https://github.com/NVIDIA-AI-IOT/deepstream_reference_apps/tree/master/deepstream-bodypose-3d) to NumPy arrays for inference.

In [7]:
!tao model pose_classification dataset_convert \
                    -e /specs/experiment_lucija.yaml \
                    results_dir=$RESULTS_DIR/nvidia \
                    encryption_key=$KEY \
                    dataset_convert.data=/data/dataset_new_json/raising.json

2026-06-08 09:13:08,953 [TAO Toolkit] [INFO] root 160: Registry: ['nvcr.io']
2026-06-08 09:13:09,008 [TAO Toolkit] [INFO] nvidia_tao_cli.components.instance_handler.local_instance 360: Running command in container: nvcr.io/nvidia/tao/tao-toolkit:6.26.3-pyt
2026-06-08 09:13:09,926 [TAO Toolkit] [INFO] nvidia_tao_cli.components.docker_handler.docker_handler 316: Printing tty value True
[2026-06-08 13:13:12,289 - TAO Toolkit - nvidia_tao_core.microservices.utils.job_utils.workflow - INFO] Logging configured at level: INFO
INFO: Logging configured at level: INFO
sys:1: UserWarning: 
'experiment_lucija.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
/usr/local/lib/python3.12/dist-packages/nvidia_tao_pytorch/core/hydra/hydra_runner.py:110: UserWarning: 
'experiment_lucija.yaml' is validated against

In [ ]:
import os
import glob
import shutil

json_folder = "/home/isaacsim/Documents/Lucija/tao-experiments/data/poseclassificationnet/dataset_new_json"
results_dir = "/home/isaacsim/Documents/Lucija/tao-experiments/poseclassificationnet/nvidia/dataset_convert"

os.makedirs(results_dir, exist_ok=True)

json_files = sorted(glob.glob(os.path.join(json_folder, "*.json")))
print(f"Pronađeno {len(json_files)} datoteka")

for json_path in json_files:
    stem = os.path.splitext(os.path.basename(json_path))[0]
    print(f"[→] Konvertiram: {stem}.json")
    
    !tao model pose_classification dataset_convert \
        -e $SPECS_DIR/experiment_lucija.yaml \
        results_dir=$RESULTS_DIR/nvidia \
        encryption_key=$KEY \
        "dataset_convert.data={json_path}"
    
    src = os.path.join(results_dir, "object_1.npy")
    dst = os.path.join(results_dir, f"{stem}.npy")
    
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f"[✓] Spremljeno: {stem}.npy")
    else:
        print(f"[✗] object_1.npy nije pronađen za {stem}")

In [13]:
 print('Converted pose data:')
 print('------------')
 !ls -lth $HOST_RESULTS_DIR/nvidia/dataset_convert

Converted pose data:
------------
total 364K
-rw-r--r-- 1 isaacsim isaacsim 359K May 12 05:07 object_1.npy
-rw-r--r-- 1 isaacsim isaacsim 3.4K May 12 05:07 status.json


This notebook has come to an end.